<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/phonchi/CryoParticleSegment/blob/main/notebook/04_particle_extraction_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

### CryoParticleSegment

In [63]:
do = False # @param{type:"boolean"}
if do:
    %pip install torchinfo -qq
    %pip install -U git+https://github.com/qubvel/segmentation_models.pytorch -qq
    %pip install starfile -qq
    %pip install https://github.com/soft-matter/trackpy/archive/master.zip -qq

In [64]:
if do:
    !git clone https://github.com/phonchi/CryoParticleSegment.git

    !wget -O /content/CryoParticleSegment/Modeling/convcrf.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/convcrf.py
    !wget -O /content/CryoParticleSegment/Modeling/dataset.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/dataset.py
    !wget -O /content/CryoParticleSegment/Modeling/model.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/model.py
    !wget -O /content/CryoParticleSegment/Modeling/trainer.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/trainer.py

In [65]:
import sys
import os

# Adjust the path relative to your current working directory
module_path = os.path.abspath('CryoParticleSegment/Modeling')

# Add to sys.path if it's not already included
if module_path not in sys.path:
    sys.path.append(module_path)

In [66]:
if do:
    #%git clone https://github.com/netw0rkf10w/CRF.git
    %cd CryoParticleSegment/Modeling/CRF_main
    !python setup.py clean --all
    !rm -rf build/
    !python setup.py build_ext --inplace --force
    !python setup.py install

> #### ⚠ Notice
>
> You need to restart the kernel after the following step.

In [67]:
if do:
    %pip install pycuda==2024.1
    %pip install "numpy<2.0"
    %pip install mrcfile -qq

## ⭐ Setup
You must run all codes under this category.

### ✅ Directory Settings

In [68]:
# @title  { display-mode: "form" }

INPUT_IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
IMAGE_DIR = INPUT_IMAGE_DIR
# @markdown ---

use_denoised_as_pariwise = False # @param {type : "boolean"}
dnzd_pw = use_denoised_as_pariwise
DENOISED_IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
# @markdown ---

LABEL_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset/10017/micrographs_ground_np" # @param {type:"string"}
DATASET_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset" # @param {type:"string"}
RESULT_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_watershed_with_02_model_map/10017/unet_eb5_dice" # @param {type:"string"}

In [69]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [70]:
# @title  { display-mode: "form" }
# @markdown Detect whether using folder in Google Drive as **`RESULT DIR`**📁.
import os
if "content" in IMAGE_DIR.split("/")[:3] or "content" in LABEL_DIR.split("/")[:3]:
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    !rm -r /content/sample_data
    if not os.path.exists("/content/image_dir"):
        if "content" in IMAGE_DIR.split("/")[:3]:
            !cp -r {IMAGE_DIR} /content/image_dir
            IMAGE_DIR = "/content/image_dir"
        if "content" in LABEL_DIR.split("/")[:3]:
            !cp -r {LABEL_DIR} /content/label_dir
            LABEL_DIR = "/content/label_dir"
  except:
    pass

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
rm: cannot remove '/content/sample_data': No such file or directory


In [71]:
IMAGE_DIR = "/content/image_dir"

In [72]:
%cd /content/

/content


### ✅ Packages Handling

In [73]:
# @title  { display-mode: "form" }
# @markdown Useful packages.

import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau, OneCycleLR

In [74]:
# @title  { display-mode: "form" }
# @markdown User-defined packages.

from dataset import MicrographDataset, MicrographDatasetEvery
from dataset import reconstruct_patched, collate_fn
from model import create_model
from trainer import CryoEMEvaluator
from trainer import CryoEMTrainerWithScheduler, tqdm_plugin_for_Trainer

## ⭐ Main

### ✅ Setting

In [75]:
# @markdown Parameters.

NUM_CLASSES = 2
EPOCHS = 50
BATCH = 8
CROP_SIZE = (512, 512)
LR = 1e-3

RLR_PATIENCE = 3
ES_PATIENCE = 15
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [76]:
# @markdown Set seed.

random_state = 42
torch.manual_seed(random_state)
torch.cuda.manual_seed_all(random_state)

## ⭐ Convcrf wtih FCN finetuned on cryoem

### ✅ Model

## The model

In [77]:
# @title  { display-mode: "form" }

architecture = "Unet++" # @param {type:"string"}
encoder = "timm-efficientnet-b5" # @param {type:"string"}
pretrained = True # @param {type:"boolean"}
solver = "fw" # @param {type:"string"}
use_unary_only = False # @param {type:"boolean"}


In [78]:
import segmentation_models_pytorch as smp

if pretrained:
  weights = "imagenet"
else:
  weights = None

if architecture == "Unet++":
    backbone = smp.UnetPlusPlus(
        encoder_name=encoder,        # choose encoder, densenet201, resnet50, e.g. mobilenet_v2 or efficientnet-b5
        encoder_weights=weights,     # use `imagenet` or `advprop` for pre-trained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
        classes=2,                      # model output channels (number of classes in your dataset)
    )

elif architecture == "Deeplab":
    backbone = smp.DeepLabV3(
        encoder_name=encoder,        # choose encoder, densenet201, resnet50, e.g. mobilenet_v2 or efficientnet-b5
        encoder_weights=weights,     # use `imagenet` pre-trained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
        classes=2,                      # model output channels (number of classes in your dataset)
    )
else:
    print("Architecture not supported")
    raise NotImplementedError

model = create_model(backbone, addout=True) #crf_args

In [79]:
import CRF
import torch.nn as nn
from model import setup_crf, create_fwcrf_model

# Example usage
solver = 'fw'  # Assuming the solver type is defined

crf = setup_crf(solver, NUM_CLASSES)
model_post = create_fwcrf_model(model.backbone, crf, use_unary_only=True)

CRF solver: fw
x0_weight: 0.0
FrankWolfeParams: 
	 scheme:	 fixed 
	 stepsize:	 1.0 (for the 'fixed' scheme) 
	 regularizer:	 l2
	 lambda_:	 1.0
	 lambda_learnable:	 False
	 x0_weight:	 0.5
	 x0_weight_learnable:	 False
Non-trainable lambda for Frank-Wolfe: 1.0
Non-trainable x0_weight for Frank-Wolfe: 0.5
Potts: remove random weights.
Add 1.0 to spatial_weight diagonal
Add 1.0 to bilateral_weight diagonal
Add -1.0 to compatibility diagonal


## ⭐ Evaluate

In [80]:
import gc
gc.collect()
torch.cuda.empty_cache()

from torchvision.utils import save_image
from dataset import reconstruct_patched

def simple_micrograph_preprocessing(micrograph):
  micrograph_copy = micrograph.copy()
  micrograph_copy = (micrograph_copy-micrograph.mean()+2.5*micrograph.std())/5/micrograph.std()
  micrograph_copy[micrograph_copy<0]=0
  micrograph_copy[micrograph_copy>1]=1
  return micrograph_copy

In [81]:
from torch.utils.data import ConcatDataset

train_dir = os.path.join(IMAGE_DIR, 'train')
train_filenames = np.loadtxt(f"{IMAGE_DIR}/train_filenames.txt", dtype=str)
if dnzd_pw == False:
    train_dataset = MicrographDatasetEvery(image_dir=train_dir, label_dir=LABEL_DIR, filenames=train_filenames, crop_size=CROP_SIZE)
else:
    dnzd_train_dir = os.path.join(DENOISED_IMAGE_DIR, 'train')
    train_dataset = MicrographDatasetEvery(image_dir=train_dir, label_dir=LABEL_DIR, denoised_dir = dnzd_train_dir, filenames=train_filenames, crop_size=CROP_SIZE)
train_loader = DataLoader(train_dataset, batch_size=None, shuffle=False, pin_memory=True, collate_fn=collate_fn)


val_dir = os.path.join(IMAGE_DIR, 'val')
val_filenames = np.loadtxt(f"{IMAGE_DIR}/val_filenames.txt", dtype=str)
if dnzd_pw == False:
    val_dataset = MicrographDatasetEvery(image_dir=val_dir, label_dir=LABEL_DIR, filenames=val_filenames, crop_size=CROP_SIZE)
else:
    dnzd_val_dir = os.path.join(DENOISED_IMAGE_DIR, 'val')
    val_dataset = MicrographDatasetEvery(image_dir=val_dir, label_dir=LABEL_DIR, denoised_dir = dnzd_val_dir, filenames=val_filenames, crop_size=CROP_SIZE)
val_loader = DataLoader(val_dataset, batch_size=None, shuffle=False, pin_memory=True, collate_fn=collate_fn)


test_dir = os.path.join(IMAGE_DIR, 'test')
test_filenames = np.loadtxt(f"{IMAGE_DIR}/test_filenames.txt", dtype=str)

test_dir = os.path.join(IMAGE_DIR, 'test')
test_filenames = np.loadtxt(f"{IMAGE_DIR}/test_filenames.txt", dtype=str)
if dnzd_pw == False:
    test_dataset = MicrographDatasetEvery(image_dir=test_dir, label_dir=None, filenames=test_filenames, crop_size=CROP_SIZE)
else:
    dnzd_test_dir = os.path.join(DENOISED_IMAGE_DIR, 'test')
    test_dataset = MicrographDatasetEvery(image_dir=test_dir, label_dir=None, denoised_dir = dnzd_test_dir, filenames=test_filenames, crop_size=CROP_SIZE)

test_loader = DataLoader(test_dataset, batch_size=None, shuffle=False, pin_memory=True, collate_fn=collate_fn)


full_filenames = np.concatenate((train_filenames, val_filenames, test_filenames))
full_dataset = ConcatDataset([train_dataset, val_dataset, test_dataset])
full_loader = DataLoader(full_dataset, batch_size=32, shuffle=True, pin_memory=True, collate_fn=collate_fn)

In [82]:
shape = None
for i1, i2, i3, i4, i5 in val_loader: #test loader and reconstruct
    shape = i5.shape
    break

In [83]:
from PIL import Image

def normalize(im):
    max_mrc=np.max(im)
    min_mrc=np.min(im)
    img_original=(255*((im-min_mrc)/(max_mrc-min_mrc))).astype(np.uint8)
    return(img_original)

def preprocess_and_crop(micrograph, crop_size=3840):
    processed_micrograph = simple_micrograph_preprocessing(micrograph)
    if crop_size:
        mic_width, mic_height = processed_micrograph.shape[1], processed_micrograph.shape[0]
        start_x, start_y = (mic_width - crop_size) // 2, (mic_height - crop_size) // 2
        end_x, end_y = start_x + crop_size, start_y + crop_size
        return processed_micrograph[start_y:end_y, start_x:end_x]
    else:
        return processed_micrograph

In [84]:
from tqdm import tqdm

fnames = []
n = len(full_dataset)
processed_micrographs = np.empty((n, shape[1], shape[2]), dtype=np.float32)

# Use tqdm to wrap your loop for a progress bar
for idx, (test_image, _, _, grid, _) in tqdm(enumerate(full_dataset), total=n, desc="Processing images"):
    name = full_filenames[idx][:-4]
    fnames.append(name)
    # Determine the directory and load the micrograph
    if os.path.exists(f"{IMAGE_DIR}/test/{name}.npy"):
        micrograph_path = f"{IMAGE_DIR}/test/{name}.npy"
    elif os.path.exists(f"{IMAGE_DIR}/train/{name}.npy"):
        micrograph_path = f"{IMAGE_DIR}/train/{name}.npy"
    else:
        micrograph_path = f"{IMAGE_DIR}/val/{name}.npy"
    micrograph = np.load(micrograph_path)
    processed_micrograph = preprocess_and_crop(micrograph)
    # Place the processed micrograph directly into the preallocated array
    processed_micrographs[idx] = processed_micrograph

Processing images: 100%|██████████| 84/84 [00:27<00:00,  3.06it/s]


In [85]:
processed_micrographs.shape

(84, 3840, 3840)

In [88]:
np.save(f"{RESULT_DIR}/processed_micrographs.npy", processed_micrographs)

In [89]:
processed_micrographs = np.load(f"{RESULT_DIR}/processed_micrographs.npy")

In [90]:
del model
torch.cuda.empty_cache()

In [91]:
model = model_post

In [92]:
RESULT_DIR

'/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_watershed_with_02_model_map/10017/unet_eb5_dice'

In [ ]:
# @title from 02 model after CRF refinement

import torch.nn.functional as F

label_images = np.empty((0, shape[1], shape[2]), dtype=np.uint8)

checkpoint_paths = [path for path in os.listdir(RESULT_DIR) if '.pt' in path]
checkpoint_path = checkpoint_paths[-1]
state_dict_path = f"{RESULT_DIR}/{checkpoint_path}"
state_dict = torch.load(state_dict_path, map_location=torch.device(device))
model.load_state_dict(state_dict, strict=False)

model.to(device)
model.eval()

mini_batch_size = 18  # Number of patches to process at once
n = len(full_dataset)
label_images = np.empty((n, shape[1], shape[2]), dtype=np.uint8)  # Preallocated array

# Iterate through the dataset with tqdm for progress tracking
for idx, (test_image, _, _, grid, _) in tqdm(enumerate(full_dataset), total=n, desc="Processing dataset"):
    # Process in batches
    with torch.no_grad():
        inputs = test_image.to(device)
        num_batches = (inputs.size(0) + mini_batch_size - 1) // mini_batch_size
        patched_outputs = []

        for batch_idx in range(num_batches):
            start_idx = batch_idx * mini_batch_size
            end_idx = min(start_idx + mini_batch_size, inputs.size(0))
            patch_input = inputs[start_idx:end_idx].to(device)
            output = model(patch_input)['out']
            patched_outputs.append(output.cpu())  # Minimize device memory usage

            # Cleanup as soon as not needed
            del patch_input, output
            torch.cuda.empty_cache()

        outputs = torch.cat(patched_outputs)  # Combine outputs
        probabilities = F.softmax(outputs, dim=1)
        class1_probabilities = probabilities[:, 1, :, :]  # Assuming class 1 is the target
        pred_image = reconstruct_patched(class1_probabilities.unsqueeze(1), grid).float()

        output_image = normalize(pred_image.squeeze().numpy())

        # Cleanup large temporary variables
        del patched_outputs, outputs, probabilities, class1_probabilities, pred_image
        torch.cuda.empty_cache()

    # Store the output image directly in the preallocated array
    label_images[idx] = output_image

    if idx % 30 == 0:
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(output_image, cmap='inferno', alpha=0.4)
        plt.show()
    del output_image
    torch.cuda.empty_cache()

In [ ]:
np.save(f"{RESULT_DIR}/label_images.npy", label_images)

In [ ]:
!cp {RESULT_DIR}/label_images.npy .

In [93]:
label_images = np.load(f"{RESULT_DIR}/label_images.npy")

In [94]:
label_images.shape

(84, 3840, 3840)

In [95]:
# @title  { vertical-output: true, display-mode: "form" }
EMPIAR_ID = 10017 # @param {type:"integer"}
RADIUS = 64 # @param {type:"integer"}
# For 10017
BORDER = 128 # @param {type:"integer"}

In [96]:
!cp {RESULT_DIR}/best_config.txt .

In [97]:
with open("best_config.txt", "r") as f:
    for line in f:
        key, value = line.strip().split(": ", 1)
        if key == "cv_config":
            cv_config = eval(value)
        elif key == "tp_config":
            tp_config = eval(value)
        elif key == "nms_config":
            nms_config = eval(value)

---

In [98]:
!cp {RESULT_DIR}/best_watershed_default_params.json .

In [99]:
import json

# Load the best parameters found
watershed_json = "best_watershed_default_params.json" # @param{type:"string"}
with open(watershed_json, 'r') as f:
    best_params = json.load(f)

# changed) Extracting specific values from the parsed dictionary
best_thresh = best_params['peak_thresh']
best_dist = best_params['min_dist']

print(f"Loaded Config: peak_Thresh={best_thresh}, min_Dist={best_dist}")

ws_config = [best_thresh, best_dist]

Loaded Config: peak_Thresh=0.4, min_Dist=0.8


---

In [100]:
!pip install starfile -qq

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/CRF-0.0.1-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [101]:
import starfile

---

In [102]:
# @title class function define
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops
import math

class WatershedDetector:
    def __init__(self, particle_radius=64, border=0, margin=40, z_thresh=3.0, use_contour_area=True):
        self.radius = particle_radius
        self.border = border
        self.margin = margin
        self.z_thresh = z_thresh
        self.use_contour_area = use_contour_area
        self.expected_area = np.pi * (self.radius ** 2)

    def _get_contour_expected_area(self, mask):
        """Calculates expected area based on actual mask contours."""
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours: return self.expected_area
        areas = [cv2.contourArea(cnt) for cnt in contours]
        median_area = np.median(areas)
        return median_area if median_area > 0 else self.expected_area

    def _calculate_metrics(self, region):
        """Generates the full dictionary of shape descriptors."""
        area = region.area
        perimeter = region.perimeter
        if perimeter == 0: return None
        return {
            'area': area,
            'circularity': (4 * np.pi * area) / (perimeter ** 2),
            'eccentricity': region.eccentricity,
            'solidity': area / region.convex_area if region.convex_area > 0 else 0,
            'hu_0': region.moments_hu[0],
            'hu_1': region.moments_hu[1],
            'hu_2': region.moments_hu[2]
        }

    def _get_mad_range(self, data):
        """Calculates the filtering window using Median Absolute Deviation."""
        arr = np.array(data)
        med = np.median(arr)
        mad = np.median(np.abs(arr - med))
        if mad == 0: return np.min(arr), np.max(arr)
        low = med - (self.z_thresh * mad / 0.6745)
        high = med + (self.z_thresh * mad / 0.6745)
        return max(0.0, low), min(1.0, high)

    def _plot_all_histograms(self, candidates, c_bounds):
        """Generates an auto-adjusting grid of histograms for all metrics. (changed)"""
        # Convert candidate metrics list of dicts to a DataFrame for easy plotting
        metrics_df = pd.DataFrame([c['metrics'] for c in candidates])
        metric_names = metrics_df.columns.tolist()

        n_metrics = len(metric_names)
        n_cols = 3
        n_rows = math.ceil(n_metrics / n_cols)

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
        axes = axes.flatten() # Flatten to 1D for easy iteration

        for idx, col_name in enumerate(metric_names):
            ax = axes[idx]
            ax.hist(metrics_df[col_name], bins=30, color='skyblue', edgecolor='black', alpha=0.7)
            ax.set_title(f"Distribution: {col_name}")
            ax.set_xlabel("Value")
            ax.set_ylabel("Frequency")

            # Only add bounds for Circularity as requested (changed)
            if col_name == 'circularity':
                ax.axvline(c_bounds[0], color='red', linestyle='--', linewidth=2, label=f'Lower: {c_bounds[0]:.2f}')
                ax.axvline(c_bounds[1], color='blue', linestyle='--', linewidth=2, label=f'Upper: {c_bounds[1]:.2f}')
                ax.legend(loc='upper right', fontsize='small')

        # Hide any unused axes in the grid
        for i in range(idx + 1, len(axes)):
            axes[i].axis('off')

        plt.tight_layout()
        plt.show()

    def detect(self, prob_map_input, peak_thresh=0.4, dist_ratio=0.8, show_plots=True):
        """Main detection entry point with optional visualization. (changed)"""
        prob_map = prob_map_input.astype(np.float32)
        if prob_map.max() > 1.0: prob_map /= 255.0
        mask = (prob_map > 0.5).astype(np.uint8)
        if np.sum(mask) == 0: mask = (prob_map > 0.05).astype(np.uint8)

        current_expected_area = self._get_contour_expected_area(mask) if self.use_contour_area else self.expected_area

        distance = ndi.distance_transform_edt(mask)
        coords = peak_local_max(distance, min_distance=int(self.radius * dist_ratio),
                                labels=mask, threshold_abs=self.radius * peak_thresh)
        markers = np.zeros(distance.shape, dtype=int)
        for i, (r, c) in enumerate(coords): markers[r, c] = i + 1
        labels = watershed(-distance, markers, mask=mask)

        candidates, circularities = [], []
        for region in regionprops(labels):
            if current_expected_area * 0.3 <= region.area <= current_expected_area * 2.0:
                m = self._calculate_metrics(region)
                if m:
                    candidates.append({'region': region, 'metrics': m})
                    circularities.append(m['circularity'])

        if not candidates: return []

        c_bounds = self._get_mad_range(circularities)

        # Trigger histogram plotting if enabled (changed)
        if show_plots:
            self._plot_all_histograms(candidates, c_bounds)

        final_particles = []
        for cand in candidates:
            if c_bounds[0] <= cand['metrics']['circularity'] <= c_bounds[1]:
                ry, rx = cand['region'].centroid
                if (self.margin < rx < prob_map.shape[1] - self.margin and
                    self.margin < ry < prob_map.shape[0] - self.margin):

                    final_particles.append({
                        'x': int(rx) + self.border,
                        'y': int(ry) + self.border,
                        'score': float(prob_map[int(ry), int(rx)]),
                        **cand['metrics']
                    })
        return final_particles

In [103]:
# @title execute watershed postprocessing

 # Control histograms
SHOW_PLOTS_A = True #@param {type:"boolean"}
# Control overlays
SHOW_PLOTS_B = True #@param {type:"boolean"}
# show plot per N micrograph to show plot
per_N_show = 30 #@param {type:"integer"}

detector = WatershedDetector(particle_radius=RADIUS, border=BORDER)
all_star_rows, all_metrics_rows = [], []

for idx, (test_image, _, _, grid, _) in enumerate(tqdm(full_dataset, desc="Processing Images")):
    fname = fnames[idx]
    label_image = label_images[idx]

    # 1. Detect particles and plot histograms (inside class)
    particles = detector.detect(label_image, peak_thresh=ws_config[0],
                                dist_ratio=ws_config[1], show_plots=SHOW_PLOTS_A and (idx % per_N_show == 0))

    if not particles: continue

    # 2. Plot Micrograph Overlay Plot
    if SHOW_PLOTS_B and idx % per_N_show == 0:
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(label_image, cmap='inferno', alpha=0.4)
        for p in particles:
            # Note: Subtracting BORDER for plot alignment if needed
            circ = plt.Circle((p['x']-BORDER, p['y']-BORDER), radius=RADIUS, fill=False, color='r')
            ax.add_patch(circ)
        ax.set_title(f"Overlay Verification: {fname} (idx: {idx})")
        plt.show()

    # 3. Data Collection
    for i, p in enumerate(particles):
        p['rlnMicrographName'] = fname + ".mrc"
        p['particle_id'] = f"{i:05d}@{fname}.mrc"
        # Explicitly add RELION coordinates to metrics DF
        p['rlnCoordinateX'] = p.pop('x')
        p['rlnCoordinateY'] = p.pop('y')
        all_metrics_rows.append(p)

        all_star_rows.append({
            'rlnCoordinateX': p['rlnCoordinateX'], 'rlnCoordinateY': p['rlnCoordinateY'],
            'rlnImageName': p['particle_id'], 'rlnMicrographName': p['rlnMicrographName']
        })

# 4. Save Final CSV
df = pd.DataFrame(all_star_rows) # For STAR file
df_metrics = pd.DataFrame(all_metrics_rows) # For Shape CSV (changed)

# Reorder columns for visibility
coord_cols = ['particle_id', 'rlnCoordinateX', 'rlnCoordinateY', 'score', 'area', 'circularity']
df_metrics = df_metrics[coord_cols + [c for c in df_metrics.columns if c not in coord_cols]]

df_metrics_sorted = df_metrics.sort_values(
    by=['rlnMicrographName', 'rlnCoordinateX', 'rlnCoordinateY'],
    ascending=True
)

parent_of_result_dir = os.path.dirname(RESULT_DIR)
csv_path = os.path.join(parent_of_result_dir, "particles_dfcsv_rst", "particle_shape_analysis.csv")
os.makedirs(os.path.join(parent_of_result_dir, "particles_dfcsv_rst"), exist_ok=True)
df_metrics_sorted.to_csv(csv_path, index=False)

print(f"Final Data Generation Complete: {len(df)} particles.")

Output hidden; open in https://colab.research.google.com to view.

In [104]:
df

,rlnCoordinateX,rlnCoordinateY,rlnImageName,rlnMicrographName
0,550,1206,00000@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
1,1771,418,00001@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
2,2266,843,00002@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
3,217,2979,00003@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
4,1711,3601,00004@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
...,...,...,...,...
40496,2131,3761,00417@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
40497,3327,2433,00418@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
40498,3668,2467,00419@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
40499,510,1133,00420@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc


In [105]:
df_metrics

,particle_id,rlnCoordinateX,rlnCoordinateY,score,area,circularity,eccentricity,solidity,hu_0,hu_1,hu_2,rlnMicrographName
0,00000@Falcon_2012_06_12-15_43_48_0.mrc,550,1206,0.996078,6539.0,0.802420,0.403076,0.946858,0.162701,0.000207,0.000103,Falcon_2012_06_12-15_43_48_0.mrc
1,00001@Falcon_2012_06_12-15_43_48_0.mrc,1771,418,0.996078,6543.0,0.854208,0.331812,0.972214,0.159913,0.000087,0.000002,Falcon_2012_06_12-15_43_48_0.mrc
2,00002@Falcon_2012_06_12-15_43_48_0.mrc,2266,843,0.996078,7389.0,0.836778,0.513757,0.965504,0.162055,0.000607,0.000011,Falcon_2012_06_12-15_43_48_0.mrc
3,00003@Falcon_2012_06_12-15_43_48_0.mrc,217,2979,0.996078,7006.0,0.849922,0.414389,0.975494,0.161619,0.000230,0.000112,Falcon_2012_06_12-15_43_48_0.mrc
4,00004@Falcon_2012_06_12-15_43_48_0.mrc,1711,3601,0.996078,6607.0,0.829399,0.191598,0.959344,0.160066,0.000009,0.000034,Falcon_2012_06_12-15_43_48_0.mrc
...,...,...,...,...,...,...,...,...,...,...,...,...
40496,00417@Falcon_2012_06_13-10_27_05_0.mrc,2131,3761,0.996078,3351.0,0.841011,0.407670,0.965706,0.163374,0.000219,0.000174,Falcon_2012_06_13-10_27_05_0.mrc
40497,00418@Falcon_2012_06_13-10_27_05_0.mrc,3327,2433,0.996078,4308.0,0.728958,0.811871,0.937745,0.184493,0.008225,0.000034,Falcon_2012_06_13-10_27_05_0.mrc
40498,00419@Falcon_2012_06_13-10_27_05_0.mrc,3668,2467,0.996078,3979.0,0.763206,0.758183,0.943114,0.177753,0.005140,0.000016,Falcon_2012_06_13-10_27_05_0.mrc
40499,00420@Falcon_2012_06_13-10_27_05_0.mrc,510,1133,0.996078,2874.0,0.818112,0.576474,0.958959,0.165041,0.001082,0.000071,Falcon_2012_06_13-10_27_05_0.mrc


In [106]:
df_metrics_sorted

,particle_id,rlnCoordinateX,rlnCoordinateY,score,area,circularity,eccentricity,solidity,hu_0,hu_1,hu_2,rlnMicrographName
3868,00149@Falcon_2012_06_12-14_33_35_0.mrc,178,1672,0.996078,5243.0,0.858086,0.479121,0.979634,0.163479,0.000449,0.000068,Falcon_2012_06_12-14_33_35_0.mrc
4168,00449@Falcon_2012_06_12-14_33_35_0.mrc,204,747,0.996078,6229.0,0.777376,0.839683,0.970098,0.190935,0.010808,0.000011,Falcon_2012_06_12-14_33_35_0.mrc
4207,00488@Falcon_2012_06_12-14_33_35_0.mrc,211,2717,0.996078,5051.0,0.786360,0.798782,0.972843,0.180903,0.007183,0.000001,Falcon_2012_06_12-14_33_35_0.mrc
4014,00295@Falcon_2012_06_12-14_33_35_0.mrc,220,3872,0.996078,5132.0,0.847173,0.671406,0.981074,0.168836,0.002414,0.000211,Falcon_2012_06_12-14_33_35_0.mrc
4147,00428@Falcon_2012_06_12-14_33_35_0.mrc,221,1374,0.996078,5823.0,0.756369,0.807020,0.961208,0.184021,0.007896,0.000069,Falcon_2012_06_12-14_33_35_0.mrc
...,...,...,...,...,...,...,...,...,...,...,...,...
40299,00220@Falcon_2012_06_13-10_27_05_0.mrc,3838,2665,0.996078,5505.0,0.851215,0.644610,0.973131,0.165844,0.001892,0.000039,Falcon_2012_06_13-10_27_05_0.mrc
40404,00325@Falcon_2012_06_13-10_27_05_0.mrc,3877,475,0.996078,5329.0,0.793571,0.652722,0.965224,0.173889,0.002216,0.000850,Falcon_2012_06_13-10_27_05_0.mrc
40199,00120@Falcon_2012_06_13-10_27_05_0.mrc,3892,638,0.996078,5599.0,0.846803,0.460372,0.975436,0.163120,0.000374,0.000202,Falcon_2012_06_13-10_27_05_0.mrc
40373,00294@Falcon_2012_06_13-10_27_05_0.mrc,3902,2570,0.996078,5386.0,0.807446,0.761880,0.962989,0.176045,0.005182,0.000017,Falcon_2012_06_13-10_27_05_0.mrc


In [107]:
# @title store star file watershed
watershed_star_name = 'watershed_particles.star' # @param{type: "string"}
starfile.write(df, f'{RESULT_DIR}/{watershed_star_name}')